# TV Series Dataset Builder

## Setup and configuration

In [ ]:
import json, time, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

try:
    from kaggle_secrets import UserSecretsClient
    TMDB_API_KEY = UserSecretsClient().get_secret("TMDB_API_KEY")
except Exception as e:
    print("Could not read Kaggle secret:", repr(e))
    TMDB_API_KEY = "PASTE_YOUR_TMDB_KEY_HERE"
OMDB_API_KEY = ""

START_YEAR, END_YEAR = 2015, 2025
MIN_IMDB_VOTES = 1000
REQUIRE_US_ORIGIN = True
KEEP_TMDB_TYPES = {"Scripted", "Miniseries"}
MIN_TMDB_VOTES = 30
PRIOR_GAP_DAYS = 180
RT_WEIGHT = 6.0
MAX_OMDB_PER_RUN = 950
TEST_LIMIT = 50

import tempfile
DATA_DIR = Path(tempfile.gettempdir()) / "imdb_data"
CACHE_DIR, OUT_DIR = Path("tmdb_cache"), Path("output")
for p in (DATA_DIR, CACHE_DIR, OUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

MIN_PRIOR_VOTES = 500
PRIOR_YEAR_GAP = 1
N_CAST, N_CREATIVE = 5, 4


## IMDb filtering

In [ ]:
IMDB_URL = "https://datasets.imdbws.com/"
EXCLUDED_GENRES = {"Reality-TV", "Talk-Show", "Game-Show", "News", "Short", "Adult"}
ENGLISH_REGIONS = {"US", "GB", "CA", "AU", "IE", "NZ"}

def download(name):
    path = DATA_DIR / name
    if path.exists():
        return path
    print(f"Downloading {name} ...")
    with requests.get(IMDB_URL + name, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            shutil.copyfileobj(r.raw, f)
    return path

def read_tsv(name, **kw):
    return pd.read_csv(download(name), sep="\t", na_values="\\N", compression="gzip", low_memory=False, **kw)

basics = read_tsv("title.basics.tsv.gz")
s = basics[basics["titleType"].isin({"tvSeries", "tvMiniSeries"})].copy()
s["startYear"] = pd.to_numeric(s["startYear"], errors="coerce")
s = s[(s["startYear"] >= START_YEAR) & (s["startYear"] <= END_YEAR) & (s["isAdult"] == 0)]
print(f"Series started {START_YEAR}-{END_YEAR}: {len(s):,}")

s = s[s["genres"].apply(lambda g: pd.isna(g) or not (set(g.split(",")) & EXCLUDED_GENRES))]
print(f"After dropping reality/talk/game/news: {len(s):,}")

ratings = read_tsv("title.ratings.tsv.gz")
s = s.merge(ratings, on="tconst", how="inner")
s = s[s["numVotes"] >= MIN_IMDB_VOTES]
print(f"With rating and >= {MIN_IMDB_VOTES} votes: {len(s):,}")

ids = set(s["tconst"]); english_ids = set()
for chunk in pd.read_csv(download("title.akas.tsv.gz"), sep="\t", na_values="\\N", compression="gzip",
                         usecols=["titleId", "region", "language"], chunksize=1_000_000):
    chunk = chunk[chunk["titleId"].isin(ids)]
    hit = chunk[(chunk["language"] == "en") | (chunk["region"].isin(ENGLISH_REGIONS))]
    english_ids.update(hit["titleId"])
candidates = s[s["tconst"].isin(english_ids)].copy()
print(f"English-language proxy: {len(candidates):,}")

if TEST_LIMIT:
    candidates = candidates.sort_values("numVotes", ascending=False).head(TEST_LIMIT)
    print(f"TEST_LIMIT active -> {len(candidates)} series")
candidates = candidates.set_index("tconst")
candidates.head()

## TMDB fetch

In [ ]:
TMDB_BASE = "https://api.themoviedb.org/3"
session = requests.Session()

def tmdb_get(path, **params):
    headers = {}
    if TMDB_API_KEY.startswith("eyJ"):
        headers["Authorization"] = f"Bearer {TMDB_API_KEY}"
    else:
        params["api_key"] = TMDB_API_KEY
    for attempt in range(5):
        try:
            r = session.get(TMDB_BASE + path, params=params, headers=headers, timeout=30)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 401:
            raise RuntimeError("TMDB rejected the API key. Check TMDB_API_KEY.")
        if r.status_code == 404:
            return None
        if r.status_code == 429:
            time.sleep(int(r.headers.get("Retry-After", 2)))
            continue
        if r.ok:
            return r.json()
        time.sleep(2 ** attempt)
    raise RuntimeError(f"TMDB request failed: {path}")

def fetch_series(imdb_id):
    out = CACHE_DIR / f"{imdb_id}.json"
    if out.exists():
        return
    found = tmdb_get(f"/find/{imdb_id}", external_source="imdb_id")
    results = (found or {}).get("tv_results", [])
    if not results:
        out.write_text(json.dumps({"_not_found": True}))
        return
    details = tmdb_get(f"/tv/{results[0]['id']}",
                       append_to_response="aggregate_credits,keywords,content_ratings,external_ids")
    if details is None:
        out.write_text(json.dumps({"_not_found": True}))
        return
    out.write_text(json.dumps(details))

def safe_fetch(imdb_id):
    try:
        fetch_series(imdb_id)
    except RuntimeError as e:
        if "API key" in str(e):
            raise
        return imdb_id
    return None

todo = [i for i in candidates.index if not (CACHE_DIR / f"{i}.json").exists()]
print(f"{len(todo):,} series still to fetch ({len(candidates) - len(todo):,} already cached)")
if TMDB_API_KEY.startswith("PASTE"):
    raise SystemExit("No TMDB key found. Kaggle: Add-ons > Secrets, add TMDB_API_KEY AND tick the box next to it for this notebook. Or paste the key into the config cell.")

with ThreadPoolExecutor(max_workers=8) as ex:
    failed = [x for x in tqdm(ex.map(safe_fetch, todo), total=len(todo)) if x]
print(f"Done. Failed (re-run this cell to retry): {len(failed)}")

## Cache backup

In [ ]:
shutil.make_archive(str(OUT_DIR / "tmdb_cache_backup"), "zip", CACHE_DIR)
print("Cache backed up to output/tmdb_cache_backup.zip")

## Parse and filter

In [ ]:
STREAMERS = {"netflix", "prime video", "amazon", "amazon prime video", "disney+", "hulu", "apple tv+", "apple tv",
             "max", "hbo max", "paramount+", "peacock", "discovery+", "freevee", "tubi", "youtube", "roku channel"}
PREMIUM = {"hbo", "showtime", "starz", "cinemax", "mgm+", "epix"}
BROADCAST = {"abc", "cbs", "nbc", "fox", "the cw", "pbs"}

def network_type(name):
    if not name:
        return "unknown"
    n = name.lower()
    if n in STREAMERS: return "streamer"
    if n in PREMIUM: return "premium_cable"
    if n in BROADCAST: return "broadcast"
    return "other"

def kw_flags(names):
    n = [k.lower() for k in names]
    return {
        "is_adaptation": int(any("based on" in k and "true story" not in k for k in n)),
        "is_true_story": int(any("true story" in k for k in n)),
        "is_remake_or_reboot": int(any(("remake" in k) or ("reboot" in k) for k in n)),
        "is_spinoff": int(any(("spin off" in k) or ("spin-off" in k) or ("spinoff" in k) for k in n)),
    }

def parse_series(d, imdb_id, base):
    seasons = d.get("seasons") or []
    s1 = next((x for x in seasons if x.get("season_number") == 1), None)
    nets = d.get("networks") or []
    cos = d.get("production_companies") or []
    kws = [k["name"] for k in (d.get("keywords") or {}).get("results", [])]
    cr = (d.get("content_ratings") or {}).get("results", [])
    age = next((r["rating"] for r in cr if r.get("iso_3166_1") == "US" and r.get("rating")), None)
    cast = sorted((d.get("aggregate_credits") or {}).get("cast", []), key=lambda c: c.get("order", 999))[:5]
    runtime = base["runtimeMinutes"]
    if pd.isna(runtime) and d.get("episode_run_time"):
        runtime = float(np.median(d["episode_run_time"]))
    net = nets[0]["name"] if nets else None
    spoken = [l.get("iso_639_1") for l in (d.get("spoken_languages") or []) if l.get("iso_639_1")]
    row = {
        "imdb_id": imdb_id, "tmdb_id": d["id"], "title": base["primaryTitle"],
        "imdb_rating": base["averageRating"], "imdb_votes": base["numVotes"],
        "tmdb_rating": d.get("vote_average"), "tmdb_votes": d.get("vote_count"),
        "title_type": base["titleType"], "start_year": int(base["startYear"]),
        "first_air_date": d.get("first_air_date"), "runtime_min": runtime,
        "genres": base["genres"].split(",") if isinstance(base["genres"], str) else [],
        "age_rating": age, "original_language": d.get("original_language"),
        "origin_country": ",".join(d.get("origin_country") or []), "tmdb_type": d.get("type"),
        "has_other_language": int(any(l != "en" for l in spoken)) if spoken else np.nan,
        "s1_episode_count": s1["episode_count"] if s1 else np.nan,
        "network": net, "network_type": network_type(net),
        "production_company": cos[0]["name"] if cos else None,
        "creator_ids": [c["id"] for c in (d.get("created_by") or [])],
        "cast_ids": [c["id"] for c in cast],
        "total_seasons": d.get("number_of_seasons"), "total_episodes": d.get("number_of_episodes"),
        "status": d.get("status"), "cast_size": len((d.get("aggregate_credits") or {}).get("cast", [])),
    }
    row.update(kw_flags(kws))
    row["is_streaming_original"] = int(row["network_type"] == "streamer")
    return row

rows, not_found = [], 0
for imdb_id, base in candidates.iterrows():
    p = CACHE_DIR / f"{imdb_id}.json"
    if not p.exists():
        continue
    d = json.loads(p.read_text())
    if d.get("_not_found"):
        not_found += 1
        continue
    rows.append(parse_series(d, imdb_id, base))
raw = pd.DataFrame(rows)
print(f"Parsed {len(raw):,} series ({not_found} not found on TMDB)")

df = raw[raw["original_language"] == "en"]
print(f"Original language = en:  {len(df):,}")
if REQUIRE_US_ORIGIN:
    df = df[df["origin_country"].fillna("").str.split(",").apply(lambda c: "US" in c)]
    print(f"US in origin_country:    {len(df):,}")
df = df[df["tmdb_type"].isin(KEEP_TMDB_TYPES)].copy()
print(f"Scripted / miniseries:   {len(df):,}")

df["premiere_date"] = pd.to_datetime(df["first_air_date"], errors="coerce")
df["premiere_date"] = df["premiere_date"].fillna(pd.to_datetime(df["start_year"].astype(str) + "-01-01"))
df["premiere_month"] = df["premiere_date"].dt.month
df.loc[df["tmdb_votes"].fillna(0) < MIN_TMDB_VOTES, "tmdb_rating"] = np.nan
df = df.reset_index(drop=True)
df[["title", "start_year", "network", "imdb_rating", "tmdb_rating"]].head()

## Rotten Tomatoes scores

In [ ]:
OMDB_CACHE = OUT_DIR / "omdb_cache.json"
rt_cache = json.loads(OMDB_CACHE.read_text()) if OMDB_CACHE.exists() else {}

def fetch_rt(imdb_id):
    j = requests.get("https://www.omdbapi.com/", params={"i": imdb_id, "apikey": OMDB_API_KEY}, timeout=30).json()
    err = (j.get("Error") or "").lower()
    if "limit" in err:
        raise RuntimeError(j["Error"])
    val = next((x["Value"] for x in j.get("Ratings", []) if x["Source"] == "Rotten Tomatoes"), None)
    return float(val.strip("%")) if val else None

if OMDB_API_KEY:
    todo_rt = [i for i in df["imdb_id"] if i not in rt_cache][:MAX_OMDB_PER_RUN]
    print(f"Fetching RT for {len(todo_rt)} series ...")
    try:
        for n, i in enumerate(tqdm(todo_rt), 1):
            rt_cache[i] = fetch_rt(i)
            if n % 50 == 0:
                OMDB_CACHE.write_text(json.dumps(rt_cache))
    except RuntimeError as e:
        print("Stopped:", e)
    OMDB_CACHE.write_text(json.dumps(rt_cache))
df["rt_critic"] = pd.to_numeric(df["imdb_id"].map(rt_cache), errors="coerce")
print(f"RT critic score available for {df['rt_critic'].notna().sum():,} of {len(df):,} series")

## Combined score

In [ ]:
def combined_score(r):
    parts = []
    if pd.notna(r["imdb_rating"]):
        parts.append((r["imdb_rating"], np.log1p(r["imdb_votes"])))
    if pd.notna(r["tmdb_rating"]):
        parts.append((r["tmdb_rating"], np.log1p(r["tmdb_votes"])))
    if pd.notna(r["rt_critic"]):
        parts.append((r["rt_critic"] / 10, RT_WEIGHT))
    if not parts:
        return np.nan
    scores, w = zip(*parts)
    return float(np.average(scores, weights=w))

df["combined_score"] = df.apply(combined_score, axis=1)
print(df[["imdb_rating", "tmdb_rating", "rt_critic", "combined_score"]].describe().round(2))

## People history features

In [ ]:
def build_long(frame, id_col):
    recs = []
    for r in frame.itertuples():
        for pid in getattr(r, id_col):
            recs.append((pid, r.imdb_id, r.premiere_date, r.combined_score))
    return pd.DataFrame(recs, columns=["person_id", "series_id", "date", "score"])

def prior_stats(long_df, gap_days):
    gap = np.timedelta64(gap_days, "D")
    rows = []
    for pid, g in long_df.sort_values("date").groupby("person_id"):
        dates = g["date"].to_numpy()
        scores = g["score"].to_numpy(dtype=float)
        sids = g["series_id"].to_numpy()
        for i in range(len(g)):
            m = dates < (dates[i] - gap)
            sc = scores[m]
            sc = sc[~np.isnan(sc)]
            rows.append((sids[i], pid, int(m.sum()),
                         sc.mean() if len(sc) else np.nan, sc.max() if len(sc) else np.nan))
    return pd.DataFrame(rows, columns=["series_id", "person_id", "prior_count", "prior_avg", "prior_best"])

def aggregate_people(stats, avg_name, count_name, best_name):
    out = stats.groupby("series_id").agg(
        a=("prior_avg", "mean"), c=("prior_count", "mean"), b=("prior_best", "max"), mx=("prior_count", "max"))
    out.columns = [avg_name, count_name, best_name, "max_count"]
    return out

creator_stats = prior_stats(build_long(df, "creator_ids"), PRIOR_GAP_DAYS)
cast_stats = prior_stats(build_long(df, "cast_ids"), PRIOR_GAP_DAYS)

cr = aggregate_people(creator_stats, "creator_prior_avg_rating", "creator_prior_count", "creator_prior_best")
cr["creator_is_debut"] = (cr.pop("max_count") == 0).astype(int)
ca = aggregate_people(cast_stats, "cast_prior_avg_rating", "cast_prior_count", "cast_max_prior_rating")
ca = ca.drop(columns="max_count")

df = df.merge(cr, left_on="imdb_id", right_index=True, how="left").merge(ca, left_on="imdb_id", right_index=True, how="left")
print(df[["creator_prior_avg_rating", "creator_prior_count", "creator_is_debut",
          "cast_prior_avg_rating", "cast_prior_count"]].describe().round(2))

## IMDb principals features

In [ ]:
titles = basics.loc[basics["titleType"].isin({"movie", "tvMovie", "tvSeries", "tvMiniSeries"}), ["tconst", "startYear"]].copy()
titles["startYear"] = pd.to_numeric(titles["startYear"], errors="coerce")
titles = titles.merge(ratings, on="tconst")
titles = titles[(titles["numVotes"] >= MIN_PRIOR_VOTES) & titles["startYear"].notna()]
keep_ids = set(titles["tconst"]) | set(df["imdb_id"])
print(f"{len(titles):,} eligible earlier titles")

CATS = {"director", "writer", "actor", "actress"}
parts = []
for chunk in tqdm(pd.read_csv(download("title.principals.tsv.gz"), sep="\t", na_values="\\N", compression="gzip",
                              usecols=["tconst", "ordering", "nconst", "category"], chunksize=2_000_000)):
    parts.append(chunk[chunk["category"].isin(CATS) & chunk["tconst"].isin(keep_ids)])
pr = pd.concat(parts, ignore_index=True)
pr["group"] = np.where(pr["category"].isin({"actor", "actress"}), "cast", "creative")
print(f"Kept {len(pr):,} principal rows")

series_year = df.set_index("imdb_id")["premiere_date"].dt.year.rename("series_year")
links = pr[pr["tconst"].isin(set(df["imdb_id"]))].sort_values(["tconst", "group", "ordering"]).copy()
links["rank"] = links.groupby(["tconst", "group"]).cumcount()
links = links[((links["group"] == "cast") & (links["rank"] < N_CAST)) |
              ((links["group"] == "creative") & (links["rank"] < N_CREATIVE))]
links = links.drop_duplicates(["tconst", "nconst", "group"]).join(series_year, on="tconst")

hist = (pr.merge(titles, on="tconst")[["nconst", "tconst", "startYear", "averageRating", "numVotes"]]
          .drop_duplicates(["nconst", "tconst"]))
hist = hist[hist["nconst"].isin(set(links["nconst"]))].rename(columns={"tconst": "prior_id"})
hist["log_votes"] = np.log1p(hist["numVotes"])

m = links[["tconst", "nconst", "group", "series_year"]].merge(hist, on="nconst", how="inner")
m = m[m["startYear"] <= m["series_year"] - PRIOR_YEAR_GAP]

person = m.groupby(["tconst", "nconst", "group"]).agg(
    p_avg=("averageRating", "mean"), p_best=("averageRating", "max"),
    p_count=("prior_id", "count"), p_lv=("log_votes", "mean")).reset_index()
pl = links[["tconst", "nconst", "group"]].merge(person, on=["tconst", "nconst", "group"], how="left")
pl["p_count"] = pl["p_count"].fillna(0)

agg = pl.groupby(["tconst", "group"]).agg(
    avg=("p_avg", "mean"), best=("p_best", "max"), count=("p_count", "mean"),
    debut=("p_count", lambda x: (x == 0).mean()), lv=("p_lv", "mean")).reset_index()
wide = agg.pivot(index="tconst", columns="group", values=["avg", "best", "count", "debut", "lv"])
wide.columns = [f"imdb_{grp}_prior_{stat}" for stat, grp in wide.columns]

df = df.drop(columns=[c for c in df.columns if c in wide.columns])
df = df.merge(wide, left_on="imdb_id", right_index=True, how="left")

new_cols = list(wide.columns)
print("\nMissing values (%): old people features vs new")
print((df[["creator_prior_avg_rating", "cast_prior_avg_rating"]].isna().mean() * 100).round(1).to_string())
print((df[new_cols].isna().mean() * 100).round(1).to_string())
df[new_cols].describe().round(2)

## Save dataset

In [ ]:
df["is_coproduction"] = (df["origin_country"].fillna("").str.count(",") >= 1).astype(int)

full = df.drop(columns=["creator_ids", "cast_ids", "tmdb_type", "first_air_date"]).copy()
full["genres"] = full["genres"].apply(lambda g: "|".join(g))
full.to_csv(OUT_DIR / "series_dataset_full.csv", index=False)

print(f"Full dataset: {full.shape} -> output/series_dataset_full.csv")
print("\nMissing values (%), top 10:")
print((full.isna().mean() * 100).round(1).sort_values(ascending=False).head(10).to_string())